In [0]:
-- Step 1: Ensure audit schema exists within the metastore catalog
CREATE SCHEMA IF NOT EXISTS maven_market_uc.audit
COMMENT 'Central Governance, Security, and Operational Telemetry Schema';

-- Step 2: Declare the centralized structured audit log Delta table
CREATE OR REPLACE TABLE maven_market_uc.audit.audit_logs (
    timestamp TIMESTAMP NOT NULL COMMENT 'UTC timestamp of the log event',
    environment STRING NOT NULL COMMENT 'Target environment: dev, staging, prod',
    pipeline_name STRING NOT NULL COMMENT 'Parent job or DLT pipeline identifier',
    step_name STRING NOT NULL COMMENT 'Granular step, task, or table module',
    status STRING NOT NULL COMMENT 'Execution status: STARTED, SUCCESS, FAILED, WARNING',
    records_processed BIGINT NOT NULL COMMENT 'Count of rows mutated, ingested, or transformed',
    error_message STRING COMMENT 'Detailed stacktrace or error description upon failure',
    metadata STRING COMMENT 'JSON string containing key-value runtime details'
)
USING DELTA
PARTITIONED BY (environment, status)
TBLPROPERTIES (
    'delta.appendOnly' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
COMMENT 'Tracks lineage, execution health, row counts, and exceptions across all pipelines';